# Module 8 — Data Visualization

**By the end of this notebook, you will be able to:**
- State the result you are communicating as a single sentence aimed at a named decision-maker
- Choose a chart type from what the message is, not from what the data happens to look like
- Strip a default figure of everything that carries no information, and ground it with an explicit, left-aligned title
- Use colour as a focusing device — checked for accessibility, not just for looks

**Context:** Modules 1 to 7 all end the same way: a table of coefficients, a mean and a standard deviation per fold, a grid of hyperparameters. Every one of those is an honest answer to a question you asked yourself. None of them is something a decision-maker can act on. This module teaches the four steps that turn a result into a figure someone else can decide from — context, chart choice, decluttering, focus — on one small, complete example, twice: once per message the same data can tell.

## Two very different kinds of analysis

**Exploratory analysis** is what you have been doing since Session 2: trying things, looking at the data from every angle, following whatever seems interesting. Nobody but you needs to see every dead end you went down.

**Explanatory analysis** is different: you have already found something, and now you need someone else — who was not in the room while you explored — to understand it and act on it. That someone has a name, a job, and about ninety seconds of attention. Every module before this one produced a *result*. This module is about the second half of the job: turning a result into something a specific person can decide from.

Three questions frame every explanatory figure, before you touch a chart:
1. **Who** is your audience?
2. **What** do you need them to know or do?
3. **How** will the data help make your point?

This module draws directly on Cole Nussbaumer Knaflic's *Storytelling with Data* and the IMT Atlantique data storytelling course (https://formations.imt-atlantique.fr/data_storytelling/) — the same four steps you will apply here.

## Capacity vs. demand

Twelve months of a home heating-and-cooling installation & repair company: its **capacity** (service hours its technicians have available) against its **demand** (service hours customers are requesting), across a single calendar year. Small enough to hold in your head, and — as you are about to see — capable of telling two entirely different stories.

In [ ]:
# Given: twelve months of capacity and demand data, one calendar year
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

months = [
    "2019-01", "2019-02", "2019-03", "2019-04", "2019-05", "2019-06",
    "2019-07", "2019-08", "2019-09", "2019-10", "2019-11", "2019-12",
]
capacity = [30500, 30100, 29700, 29263, 28037, 21596, 25895, 25813, 22427, 23605, 24263, 24243]
demand = [39500, 41800, 43900, 46193, 49131, 50124, 48850, 47602, 43697, 41058, 37364, 34364]

capacity_df = pd.DataFrame({"month": months, "capacity": capacity, "demand": demand})
capacity_df

### The chart your tool gave you

Before touching anything, here is what one line of `pandas` gives you for two monthly series, with no styling at all — this is a genuine default, not something built to look bad on purpose. It is the starting point every step below will change, one at a time.

In [ ]:
# Given: the default chart — one line of code, nothing styled
capacity_df.plot(x="month", y=["capacity", "demand"], kind="bar", title="Capacity and Demand by Month")
plt.show()

One thing you might have expected and do not see: gridlines. Many tools (Excel, `seaborn`'s default style) add them automatically — plain `matplotlib` does not, and has not since its 2017 style overhaul (`plt.rcParams["axes.grid"]` is `False` by default, checked directly in this environment). So the usual "step 1: remove the gridlines" advice has nothing to do here — this chart's clutter is elsewhere: the spines, the legend box, the 90° tick labels, and the raw `"2019-01"` strings standing in for a month name. If you are ever working in a tool or theme that does turn gridlines on by default, removing them still belongs at Step 3.

## Step 1 — Understand the context

Before drawing anything, decide **who** you are talking to and **what** you need them to do. The same twelve numbers above support two entirely different messages — both are given below, verbatim, from the Big Idea worksheet:

> **Big Idea #1:** We need to recruit staff during the summer period (demand is highest when production is lowest)

> **Big Idea #2:** We are not able to meet demand throughout the year, let's invest in our production infrastructure

Both are true. Neither is "the" answer — the audience decides which one you tell. Big Idea #1 is aimed at whoever approves seasonal hiring; Big Idea #2 is aimed at whoever approves capital investment. A Big Idea should (1) articulate a point of view, (2) convey what's at stake, and (3) be a single, complete sentence.

**Big Idea #1 is built for you below, at every step, as a worked example. Big Idea #2 is yours to build — same step, right after, on your own.**

**Documentation references:**
- [1. Understand the Context](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_8/#1-understand-the-context) on the course site — the Who/What/How framework and the Big Idea criteria in full

## Step 2 — Choose an appropriate visual

Two rules to keep in mind, from the "three things to remember": avoid pie charts and 3D graphics, and always use a single vertical axis with a zero baseline. A short reference for what is actually available — `matplotlib` and `seaborn`/`pandas` both included, use whichever you prefer:

| Chart type | `matplotlib` | `seaborn` / `pandas` |
|---|---|---|
| Line | [`ax.plot(x, y)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html) | [`sns.lineplot(data=df, x=..., y=...)`](https://seaborn.pydata.org/generated/seaborn.lineplot.html) / `df.plot(kind="line")` |
| Grouped bar | [`ax.bar(x, y, width=...)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar.html) | [`sns.barplot(...)`](https://seaborn.pydata.org/generated/seaborn.barplot.html) / `df.plot(kind="bar")` |
| Filled area | [`ax.fill_between(x, y1, y2)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html) | — |

**Big Idea #1's chart:** this message needs both series visible together across the whole year, so you can later point at a specific stretch of months. Here is a line chart of `capacity` and `demand`, plotted against real dates (`pd.to_datetime(capacity_df["month"])`) rather than the raw strings — a real date axis is what lets you point at a date range later.

**Documentation references:**
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [2. Choose an Appropriate Visual](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_8/#2-choose-an-appropriate-visual) on the course site — the full chart taxonomy and tool links

In [ ]:
# Given: Big Idea #1's chart — a line for capacity, a line for demand, against real dates
capacity_df["date"] = pd.to_datetime(capacity_df["month"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(capacity_df["date"], capacity_df["capacity"], label="Capacity")
ax.plot(capacity_df["date"], capacity_df["demand"], label="Demand")
ax.legend()
ax.set_title("Capacity and Demand by Month")
plt.show()

**Exercise — Big Idea #2:** this message is not about the two raw series — it is about the gap between them. Compute `capacity_df["gap"] = capacity_df["demand"] - capacity_df["capacity"]`, and plot that single number as one line. One curve, one message.

In [ ]:
# TODO 1: Big Idea #2's chart — one line, the monthly gap between demand and capacity


## Step 3 — Eliminate clutter

At this step, change **only the habillage** — spines, legend, tick labels, and *where* the title sits — not the colors and not what the title says (that is Step 4). The clutter this course warns about is specific: *"borders, gridlines, data markers, and the like"* — elements that make the audience do work without adding information.

**Big Idea #1's chart, decluttered:** the top and right spines are gone, the legend has no border. Every month still gets its own tick — nothing thinned out — but each one is relabeled with the first three letters of its month name, capitalized (`JAN`, `FEB`, `MAR`, …), instead of the raw `"2019-01"` string; since the whole year is 2019, the year appears once, as the x-axis label, instead of twelve times. The title is left-aligned with `loc="left"` and a bit larger than the default — still descriptive for now, Step 4 rewrites what it says.

**Documentation references:**
- [`matplotlib.axes.Axes.spines`](https://matplotlib.org/stable/api/spines_api.html)
- [`Axes.set_title(loc=...)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_title.html)
- [3. Eliminate Clutter](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_8/#3-eliminate-clutter) on the course site — all eight Gestalt principles

In [ ]:
# Given: Big Idea #1's chart, decluttered — habillage only, no color or title-wording changes
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(capacity_df["date"], capacity_df["capacity"], label="Capacity")
ax.plot(capacity_df["date"], capacity_df["demand"], label="Demand")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
ax.set_xticks(capacity_df["date"])
ax.set_xticklabels([d.strftime("%b").upper() for d in capacity_df["date"]])
ax.set_xlabel("2019")
ax.set_ylabel("Hours")
ax.set_title("Capacity and Demand by Month", loc="left", fontsize=15)
plt.show()

**Exercise — Big Idea #2:** the same moves, on one line instead of two.

In [ ]:
# TODO 2: declutter Big Idea #2's chart — the same habillage moves, on one line


## Step 4 — Draw attention where you want it

Before choosing any color, check it against common color-vision deficiencies — [David Nichols' colorblindness simulator, pre-loaded with the IBM Design Library palette](https://davidmathlogic.com/colorblind/#%23648FFF-%23785EF0-%23DC267F-%23FE6100-%23FFB000), simulating deuteranopia, protanopia and more in one view. This module uses that palette throughout — `#648FFF` (blue), `#785EF0` (purple), `#DC267F` (pink), `#FE6100` (orange), `#FFB000` (gold) — instead of matplotlib's own defaults, which measurably lose more separation under the same simulation: matplotlib's blue/orange (`#1f77b4`/`#ff7f0e`) drops from 1.09 (normal vision) to 0.76–0.85 under protanopia/deuteranopia simulation, while IBM's blue/orange (`#648FFF`/`#FE6100`) stays at 1.02–1.03 — a real, measured improvement, not just a different look.

Grey plus one accent is the usual move — but Big Idea #1 needs both series individually identifiable throughout, so greying one out is not the right technique here. Instead, the focus device is a **faded-vs-vivid line** plus **the area between the two curves**, restricted to the window that actually matters.

**Big Idea #1's chart, focused:** `Capacity` is recolored `#648FFF`, `Demand` is `#FE6100`. The message is about one specific stretch — June through September — while the rest of the year is context, not the point:
- Each line is plotted twice: once across the whole year at low `alpha` (faded), then again for just the June–September slice at full `alpha` (vivid) — the second call draws on top of the first, so no seam shows.
- The area **between the two curves** is filled — not a flat background rectangle — restricted to that same June–September slice, so the gap itself becomes part of what is highlighted.
- Each of the four months inside the window is marked on both curves and labeled with its value, in that series' own color.

The plain title is replaced with one where the words "Capacity" and "Demand" are colored to match their lines, using `colored_title` below — which also means the separate legend is gone.

**Documentation references:**
- [`Axes.fill_between`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html), [`Axes.annotate`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html)
- [4. Draw Attention Where You Want It](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_8/#4-draw-attention-where-you-want-it) on the course site — preattentive attributes and the accessibility check

In [ ]:
# Given: draw a title built from several differently colored pieces (the "rainbow text" technique)
from matplotlib.offsetbox import AnnotationBbox, HPacker, TextArea


def colored_title(ax, parts, fontsize=16, y=1.03):
    """
    Draw a left-aligned title built from several differently colored pieces,
    so a reader can read color-coded names directly in the title instead of
    looking them up in a separate legend.

    Parameters:
    - ax: the Axes to place the title above
    - parts: list of (text, color) tuples, drawn left to right
    - fontsize: font size applied to every piece
    - y: vertical position of the title, in axes fraction

    Returns:
    - the AnnotationBbox artist holding the composed title
    """
    boxes = [
        TextArea(text, textprops=dict(color=color, fontsize=fontsize, fontweight="bold"))
        for text, color in parts
    ]
    packed = HPacker(children=boxes, align="baseline", pad=0, sep=2)
    box = AnnotationBbox(
        packed, (0, y), xycoords=("axes fraction", "axes fraction"),
        box_alignment=(0, 0), frameon=False,
    )
    ax.add_artist(box)
    return box

In [ ]:
# Given: Big Idea #1's chart, focused — faded-vs-vivid lines, a between-curves fill, labeled zone months
capacity_color = "#648FFF"
demand_color = "#FE6100"
zone_start, zone_end = pd.Timestamp("2019-06-01"), pd.Timestamp("2019-09-30")
in_zone = (capacity_df["date"] >= zone_start) & (capacity_df["date"] <= zone_end)
zone = capacity_df[in_zone]

fig_bg1, ax = plt.subplots(figsize=(9.5, 5.5))
# faded, full-year lines
ax.plot(capacity_df["date"], capacity_df["capacity"], color=capacity_color, alpha=0.3)
ax.plot(capacity_df["date"], capacity_df["demand"], color=demand_color, alpha=0.3)
# vivid, zone-only overlay
ax.plot(zone["date"], zone["capacity"], color=capacity_color, linewidth=2.2)
ax.plot(zone["date"], zone["demand"], color=demand_color, linewidth=2.2)
# the gap itself, filled, zone only
ax.fill_between(zone["date"], zone["capacity"], zone["demand"], color="grey", alpha=0.15)

# Capacity dips low in June and September but stays relatively flat and
# closer to the fill in July and August — label those two above the line
# instead of below, so the text never sits on top of the curve.
capacity_label_above = {"Jul", "Aug"}
demand_label_above = {"Jun", "Jul", "Aug"}

for _, row in zone.iterrows():
    month = row["date"].strftime("%b")
    capacity_offset = 5 if month in capacity_label_above else -5
    capacity_va = "bottom" if month in capacity_label_above else "top"
    demand_offset = 5 if month in demand_label_above else -5
    demand_va = "bottom" if month in demand_label_above else "top"

    ax.scatter(row["date"], row["capacity"], color=capacity_color, zorder=5)
    ax.annotate(f"{row['capacity']:,.0f}", (row["date"], row["capacity"]),
                textcoords="offset points", xytext=(0, capacity_offset), color=capacity_color,
                ha="center", va=capacity_va, fontsize=9, fontweight="bold")
    ax.scatter(row["date"], row["demand"], color=demand_color, zorder=5)
    ax.annotate(f"{row['demand']:,.0f}", (row["date"], row["demand"]),
                textcoords="offset points", xytext=(0, demand_offset), color=demand_color,
                ha="center", va=demand_va, fontsize=9, fontweight="bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks(capacity_df["date"])
ax.set_xticklabels([d.strftime("%b").upper() for d in capacity_df["date"]])
ax.set_xlabel("2019")
ax.set_ylabel("Hours")
colored_title(ax, [
    ("Hire seasonal technicians — ", "black"), ("Capacity", capacity_color),
    (" won't cover ", "black"), ("Demand", demand_color), (" this summer", "black"),
])
plt.show()

**The "close your eyes" test:** close your eyes, then open them and look at Big Idea #1's chart for one second. What do you see first? If it is not the June–September stretch and the two names in the title, the focus is not strong enough yet.

**Exercise — Big Idea #2:** there is no single window to highlight here — every month is in deficit, which is the whole point, so accent the *entire* curve instead of one stretch of it, and fill the area under it down to zero, so "never enough" reads as a shape, not just a line. Pick a color this notebook has not used yet: orange (`#FE6100`) already means "Demand" in Big Idea #1's chart, so reusing it here would suggest a connection that is not there. Use `#DC267F` (pink), from the same IBM palette, instead. Since the message is about the whole year rather than any one month, drop the tick marks entirely this time — keep only `2019` as the x-axis label — and finish the title by naming the year.

In [ ]:
# TODO 3: focus Big Idea #2's chart — a fresh accent color, filled under the curve, a takeaway title


**The "close your eyes" test, again:** the same test, on Big Idea #2's chart — is the very first thing you notice that the filled shape never touches zero?

### From default to a decision — what changed, and why

- **Step 1 (context):** two audiences, two Big Ideas, from the same twelve numbers.
- **Step 2 (visual):** Big Idea #1 needed both raw series, visible together — a line chart. Big Idea #2 needed neither series on its own — a single line of the computed gap.
- **Step 3 (clutter):** spines and a framed legend were removed; every month kept its own tick, relabeled to three letters, with the year moved to the axis label instead of being repeated twelve times; the title moved to the left and got a size bump.
- **Step 4 (focus):** colors were chosen only after checking they hold up under color-vision deficiencies (IBM's palette measurably better than matplotlib's defaults here), and no color was reused between the two charts. Big Idea #1 faded the twelve months down to the four that matter, filled the gap between the curves only there, and labeled those four points directly. Big Idea #2 filled its entire curve, because *every* month is the point, and dropped its ticks entirely since no single month needed pointing at. Both titles now carry their own legend, in color.

No single step would have gotten either chart there alone — Step 2's choice decided what Step 4 would even have to draw attention to.

## Export both charts

You need both finished charts for your Reflection answer on the site. Big Idea #1's is exported below as another given step; Big Idea #2's is yours to write.

In [ ]:
# Given: save Big Idea #1's chart
import os

os.makedirs("../../reflection/session_3", exist_ok=True)
fig_bg1.savefig("../../reflection/session_3/module_8_BG1.png", dpi=150, bbox_inches="tight")

In [ ]:
# TODO 4: save Big Idea #2's chart


**Continue on the site:** [Module 8 — Data Visualization](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_8/)